# Notebook 04c — RL Sanity Check (N=1, SPY)

**Purpose**: end-to-end contract verification for the Session 4a scaffold.
An _untrained_ RLAgent (random policy initialization) is run on SPY alone
through the existing `run_horse_race` harness.

Passing criteria:
- No exceptions from env, policy, agent, strategy, or harness
- Final Sharpe is nowhere near 2.579 (ML bar) — confirms no evaluation bug
- Turnover ≈ 0 (N=1 softmax is always `[1.0]`, no rebalancing)

In [ ]:
import sys
from pathlib import Path

# Colab: install package from GitHub if running remotely
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)
else:
    repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    if str(repo_root / 'src') not in sys.path:
        sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd
import numpy as np
import torch

from aiam.data.panel import Panel
from aiam.rl.env import PortfolioEnv
from aiam.rl.policy import SimplexPolicy
from aiam.rl.agent import RLAgent
from aiam.rl.strategy import RLStrategy
from aiam.harness.horse_race import run_horse_race
from aiam.evaluation.performance import performance_stats

print('Imports OK')

## §1 — Load SPY prices, build N=1 Panel

In [ ]:
LOOKBACK = 20
TEST_START = '2023-01-01'
ASSET = 'SPY.US'

# Resolve data path for both local and Colab execution
data_dir = Path('data/cache') if Path('data/cache').exists() else Path('../data/cache')

prices_all = pd.read_parquet(data_dir / 'prices_29.parquet')
spy_prices = prices_all[[ASSET]].dropna()

panel = Panel({'prices': spy_prices})

print(f'SPY prices: {spy_prices.shape[0]} rows, {spy_prices.index[0].date()} → {spy_prices.index[-1].date()}')
print(f'Test window start: {TEST_START}')
print(f'Universe at {TEST_START}: {panel.universe_at(TEST_START)}')

## §2 — Build PortfolioEnv and verify state shape

In [ ]:
spy_returns = spy_prices.pct_change().dropna()

env = PortfolioEnv(spy_returns, lookback=LOOKBACK)
state = env.reset()

print(f'Env  : N={env.n_assets} asset(s), F={env.n_features} features, T={len(spy_returns)} days')
print(f'State: features={state["features"].shape}, weights={state["weights"]}')

# Quick episode roll: verify step returns finite values
w = np.array([1.0], dtype=np.float32)  # 100% SPY
_, reward, done, info = env.step(w)
print(f'Step  : reward={reward:.6f}, turnover={info["turnover"]:.6f}, done={done}')
assert np.isfinite(reward), 'reward must be finite'
print('Env contract OK ✓')

## §3 — Instantiate untrained RLAgent and RLStrategy

In [ ]:
torch.manual_seed(42)

policy = SimplexPolicy(n_features=LOOKBACK, hidden_dim=32)
agent = RLAgent(policy, lookback=LOOKBACK, seed=42)
strategy = RLStrategy(agent)

n_params = sum(p.numel() for p in policy.parameters())
print(f'SimplexPolicy params: {n_params:,}')

# Spot-check predict_weights at a known date
asof = pd.Timestamp(TEST_START)
w_spot = strategy.predict_weights(panel, asof)
print(f'predict_weights at {asof.date()}: {w_spot.to_dict()}')
assert abs(w_spot.sum() - 1.0) < 1e-5, 'weights must sum to 1'
assert not w_spot.isna().any(), 'no NaNs allowed'
print('Strategy contract OK ✓')

## §4 — Run harness: walk forward over test window

In [ ]:
test_end = str(spy_prices.index[-1].date())

result = run_horse_race(panel, strategy, start=TEST_START, end=test_end)

stats = result['stats']
port_returns = result['portfolio_returns'].dropna()
weights_df = result['weights']

print(f'Test period : {TEST_START} → {test_end}')
print(f'Days evaluated: {len(port_returns)}')
print()

## §5 — Results: Sharpe + turnover

In [ ]:
ML_BAR = 2.579

print('=== N=1 SPY-only sanity (UNTRAINED random policy) ===')
print(f"Sharpe ratio       : {stats['sharpe_ratio']:.4f}")
print(f"Annualized return  : {stats['annualized_return']:.2%}")
print(f"Annualized vol     : {stats['annualized_volatility']:.2%}")
print(f"Max drawdown       : {stats['max_drawdown']:.2%}")
print()

# Turnover: N=1 softmax([x]) = [1.0] always → policy never rebalances
daily_turnover = weights_df.diff().abs().sum(axis=1)
print(f'Daily turnover (mean): {daily_turnover.mean():.6f}')
print(f'Daily turnover (max) : {daily_turnover.max():.6f}')
print()

gap = ML_BAR - stats['sharpe_ratio']
print(f'Gap to ML bar (2.579): {gap:.4f}')

# Sanity guard: if Sharpe is implausibly close to the ML bar, the harness is mis-wired.
if abs(stats['sharpe_ratio'] - ML_BAR) < 0.05:
    raise RuntimeError(
        f"Sharpe {stats['sharpe_ratio']:.4f} is suspiciously close to ML bar {ML_BAR} — "
        "evaluation harness may be mis-wired. Investigate before committing."
    )

print('✓ Sharpe is clearly different from ML bar — harness wiring confirmed.')
print('✓ End-to-end RL scaffold contract verified.')